# Chapter 02 — Data Cleaning: Live Examples

**Session 1 | Chapter 2 | Duration: ~15 min**

This notebook demonstrates all data cleaning techniques with a realistic messy dataset.
We simulate a **student survey dataset** with intentional messiness.

**Key principle:** We split the data **before** any statistical transformation to avoid **data leakage**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(42)

print('Libraries loaded!')

## 1. Creating a Messy Dataset

In real life, you get this data from a CSV, database, or API.  
Here we create it manually so we know exactly what's wrong.

In [ ]:
# Simulate a messy student dataset
raw_data = {
    'student_id': range(1, 21),
    'age': [22, 25, 23, None, 21, 999, 24, 22, 26, 23,
            20, 25, None, 22, 24, 21, 23, 25, 22, 24],
    'gender': ['Male', 'female', 'MALE', 'Female', None, 'male', 'Female', 'Male',
               'M', 'Female', 'male', 'Female', 'Male', None, 'female', 'Male',
               'Female', 'male', 'FEMALE', 'Male'],
    'study_hours_per_week': [15, 20, None, 18, 25, 12, 30, 22, None, 16,
                              19, 28, 14, 21, 17, None, 23, 26, 18, 150],  # 150 = outlier
    'grade': ['A', 'B', 'A', 'C', 'B', None, 'A', 'B', 'C', 'A',
               'B', 'A', 'C', 'B', None, 'A', 'B', 'C', 'A', 'B'],
    'faculty': ['Science', 'Arts', 'Science', 'Engineering', 'Arts',
                 'Science', None, 'Arts', 'Engineering', 'Science',
                 'Arts', 'Science', 'Engineering', 'Arts', 'Science',
                 'Engineering', 'Arts', 'Science', 'Arts', 'Engineering'],
    'score': [82, 75, 91, 68, 78, 55, 95, 80, 72, 88,
               76, 92, 65, 79, 83, 70, 84, 77, 90, 73]
}

df = pd.DataFrame(raw_data)
print('Raw dataset shape:', df.shape)
df.head(10)

## 2. Inspection — Always Start Here

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = df.isnull().mean().round(3) * 100
missing_report = pd.DataFrame({'count': missing, 'percent': missing_pct})
print(missing_report[missing_report['count'] > 0])

In [ ]:
# Visual missing-value map
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False,
            cmap='Blues', ax=ax)
ax.set_title('Missing Value Map (blue = missing)', fontsize=13)
ax.set_xlabel('Columns')
plt.tight_layout()
plt.show()

## 3. Fixing the `gender` Column — Inconsistent Categories

Problem: 'Male', 'male', 'MALE', 'M' all mean the same thing.

This is a **deterministic text fix** — no statistics involved, so it's safe to do before splitting.

In [ ]:
print('Unique gender values before cleaning:')
print(df['gender'].unique())

# Step 1: Standardize to lowercase and strip whitespace
df['gender'] = df['gender'].str.lower().str.strip()

# Step 2: Map shorthand 'm' → 'male'
df['gender'] = df['gender'].replace({'m': 'male', 'f': 'female'})

print('\nUnique gender values after cleaning:')
print(df['gender'].unique())

## 4. Fix Obviously Erroneous Values

Age = 999 is clearly a data entry error, not a statistical outlier.  
We set it to `NaN` so it gets imputed properly later.

In [ ]:
# age=999 is impossible — mark as missing for imputation
df.loc[df['age'] > 100, 'age'] = np.nan
print(f'age after fixing impossible values: {df["age"].describe().round(1)}')

## 5. Train / Test Split — BEFORE Any Statistical Transformation

⚠️ **This is the most important step in the entire cleaning process.**

Everything that computes statistics from data (mean, median, IQR, scaler parameters)
must be **fit on the training set only**. Otherwise we get **data leakage** — the model
indirectly "sees" information from the test set during training.

**Rule:** Split first, then preprocess. Fit on train, transform both.

In [ ]:
from sklearn.model_selection import train_test_split

target_col = 'score'
y = df[target_col]
X = df.drop(columns=[target_col, 'student_id'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f'Training set:  {X_train.shape[0]} samples')
print(f'Test set:      {X_test.shape[0]} samples')
print(f'\nThe test set is now locked away.')
print('All following steps: fit on X_train, transform X_train AND X_test.')

## 6. Handling Missing Values (fit on train only!)

We compute median/mode **from the training set** and use those values to fill both train and test.

In [ ]:
# --- Numerical: fill with median from TRAINING set ---
for col in ['age', 'study_hours_per_week']:
    train_median = X_train[col].median()
    X_train[col] = X_train[col].fillna(train_median)
    X_test[col] = X_test[col].fillna(train_median)  # same value!
    print(f'{col}: filled with train median = {train_median}')

# --- Categorical: fill with mode from TRAINING set ---
for col in ['gender', 'grade', 'faculty']:
    train_mode = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(train_mode)
    X_test[col] = X_test[col].fillna(train_mode)  # same value!
    print(f'{col}: filled with train mode = "{train_mode}"')

print(f'\nMissing values remaining — train: {X_train.isnull().sum().sum()}, test: {X_test.isnull().sum().sum()}')

## 7. Detecting and Treating Outliers (based on train statistics)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['age', 'study_hours_per_week']):
    ax.boxplot(X_train[col], patch_artist=True,
               boxprops=dict(facecolor='#3498db', alpha=0.6))
    ax.set_title(f'Boxplot: {col} (train)', fontsize=12)
    ax.set_ylabel(col)

axes[2].boxplot(y_train, patch_artist=True,
                boxprops=dict(facecolor='#3498db', alpha=0.6))
axes[2].set_title('Boxplot: score (train)', fontsize=12)
axes[2].set_ylabel('score')

plt.suptitle('Boxplots — Outlier Detection (training data only)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# IQR-based outlier detection for study_hours — computed on TRAIN set
Q1 = X_train['study_hours_per_week'].quantile(0.25)
Q3 = X_train['study_hours_per_week'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f'Q1={Q1}, Q3={Q3}, IQR={IQR}')
print(f'Outlier bounds (from train): [{lower_bound:.1f}, {upper_bound:.1f}]')

# Clip both train and test using train-derived bounds
X_train['study_hours_per_week'] = X_train['study_hours_per_week'].clip(lower=lower_bound, upper=upper_bound)
X_test['study_hours_per_week'] = X_test['study_hours_per_week'].clip(lower=lower_bound, upper=upper_bound)

print(f'\nAfter clipping — train max: {X_train["study_hours_per_week"].max()}, test max: {X_test["study_hours_per_week"].max()}')

## 8. Encoding Categorical Features

ML models need numbers. Let's convert our categorical columns.

Binary and ordinal encoding are deterministic mappings — no leakage risk.  
One-hot encoding is also safe as long as we handle unseen categories.

In [ ]:
# --- Binary encoding: gender ---
X_train['gender_encoded'] = (X_train['gender'] == 'female').astype(int)
X_test['gender_encoded'] = (X_test['gender'] == 'female').astype(int)

# --- Ordinal encoding: grade (has natural order: C < B < A) ---
grade_order = {'C': 0, 'B': 1, 'A': 2}
X_train['grade_encoded'] = X_train['grade'].map(grade_order)
X_test['grade_encoded'] = X_test['grade'].map(grade_order)
print('Grade encoding:', grade_order)

In [ ]:
# --- One-hot encoding: faculty (nominal, no natural order) ---
# Use pd.get_dummies on train, then align test columns to match
X_train = pd.get_dummies(X_train, columns=['faculty'], prefix='faculty', dtype=int)
X_test = pd.get_dummies(X_test, columns=['faculty'], prefix='faculty', dtype=int)

# Align columns: test may have missing or extra categories
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print('One-hot encoded columns:', [c for c in X_train.columns if c.startswith('faculty_')])

In [ ]:
# Drop original text columns (no longer needed)
cols_to_drop = ['gender', 'grade']
X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

print('Final feature columns:', list(X_train.columns))
print(f'Shape — train: {X_train.shape}, test: {X_test.shape}')

## 9. Feature Scaling (fit on train only!)

The scaler learns mean and std from the training set, then transforms both sets.

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_features = ['age', 'study_hours_per_week']

print('=== Before Scaling (train) ===')
print(X_train[numerical_features].describe().round(2))

# Fit on train, transform both
scaler = StandardScaler()
X_train[numerical_features] = scaler.fit_transform(X_train[numerical_features])  # fit + transform
X_test[numerical_features] = scaler.transform(X_test[numerical_features])         # transform only!

print('\n=== After StandardScaler (train) ===')
print(X_train[numerical_features].describe().round(3))

In [ ]:
# Visual comparison: before vs after scaling (training set)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, col in enumerate(numerical_features):
    axes[i].hist(X_train[col], bins=8, color='#e74c3c', alpha=0.7, edgecolor='white')
    axes[i].set_title(f'{col} (standardized)', fontsize=11)
    axes[i].set_ylabel('Count')

plt.suptitle('Scaled Features (Training Set)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print('Note: the SHAPE of the distribution is unchanged.')
print('Only the scale (axis values) changed.')

## Summary

We went from a **messy raw dataset** to **model-ready data** in the correct order:

| Step | What we did | Leakage risk? |
|------|------------|---------------|
| 1. Inspect | Shape, dtypes, missing values | No — just looking |
| 2. Fix text inconsistencies | Standardized 'gender' values | No — deterministic |
| 3. Fix impossible values | age=999 → NaN | No — domain knowledge |
| 4. **Train/test split** | **80% train, 20% test** | **← Split point** |
| 5. Handle missing values | Median/mode **from train** | ⚠️ Would leak if computed on all data |
| 6. Remove/fix outliers | IQR bounds **from train** | ⚠️ Would leak if computed on all data |
| 7. Encode categories | Binary, ordinal, one-hot | No — deterministic mappings |
| 8. Scale features | StandardScaler **fit on train** | ⚠️ Would leak if fit on all data |

**Golden rule:** `fit()` on train only → `transform()` on both train and test.

→ Now open the **exercises** and try it yourself!